In [1]:
DATASET_ID = 'dair-ai/emotion'
CHECKPOINT = 'bert-base-uncased'

In [2]:
from datasets import load_dataset, load_dataset_builder

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

In [4]:
dataset_info = load_dataset_builder(DATASET_ID)

In [5]:
dataset_info.info.features

{'text': Value('string'),
 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}

In [6]:
dataset_info.info.splits

{'train': SplitInfo(name='train', num_bytes=1743533, num_examples=16000, dataset_name='emotion'),
 'validation': SplitInfo(name='validation', num_bytes=214945, num_examples=2000, dataset_name='emotion'),
 'test': SplitInfo(name='test', num_bytes=217423, num_examples=2000, dataset_name='emotion')}

In [7]:
dataset = load_dataset(DATASET_ID)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [8]:
dataset['train'][:5]

{'text': ['i didnt feel humiliated',
  'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake',
  'im grabbing a minute to post i feel greedy wrong',
  'i am ever feeling nostalgic about the fireplace i will know that it is still on the property',
  'i am feeling grouchy'],
 'label': [0, 0, 3, 2, 3]}

In [9]:
df = dataset['train'].to_pandas()
df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [10]:
df.isna().sum()

text     0
label    0
dtype: int64

In [11]:
df['label'].value_counts()

label
1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64

In [12]:
df.drop_duplicates()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,1
15998,i feel like this was such a rude comment and i...,3


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=6)
collator = DataCollatorWithPadding(tokenizer)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 706.58it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [14]:
def tokenize(example, tokenizer):
    return tokenizer(
        example['text'],
        truncation=True
    )

In [16]:
tokenized_dataset = dataset.map(tokenize, batched=True, fn_kwargs={'tokenizer': tokenizer})

Map: 100%|██████████| 2000/2000 [00:00<00:00, 36198.83 examples/s]


In [18]:
train_args = TrainingArguments(
    output_dir='./results-practice',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    num_train_epochs=10,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16
)

In [22]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels,
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro",
    )

    return {
        "accuracy": accuracy["accuracy"],
        "macro_f1": f1["f1"],
    }

In [23]:
trainer = Trainer(
    args=train_args,
    model=model,
    data_collator=collator,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    compute_metrics=compute_metrics
)

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.135956,0.235893,0.932500,0.909405
2,0.111575,0.192280,0.935500,0.906867
3,0.083742,0.250318,0.935000,0.909803
4,0.025626,0.294892,0.935000,0.911373
5,0.019836,0.383528,0.931000,0.902016
6,0.025432,0.325411,0.937000,0.911572
7,0.007661,0.351091,0.940500,0.918494
8,0.015663,0.406794,0.942000,0.921095


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=8000, training_loss=0.06330455033574253, metrics={'train_runtime': 828.9658, 'train_samples_per_second': 154.409, 'train_steps_per_second': 9.651, 'total_flos': 3092669078700672.0, 'train_loss': 0.06330455033574253, 'epoch': 8.0})

In [25]:
history = pd.DataFrame(trainer.state.log_history)

# Evaluation rows only
history[history["eval_loss"].notna()]

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_macro_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
20,NaN,NaN,NaN,1.0,1000,0.235893,0.9325,0.909405,3.6001,555.538,34.721,NaN,NaN,NaN,NaN,NaN
41,NaN,NaN,NaN,2.0,2000,0.192280,0.9355,0.906867,3.4131,585.978,36.624,NaN,NaN,NaN,NaN,NaN
62,NaN,NaN,NaN,3.0,3000,0.250318,0.9350,0.909803,3.4478,580.080,36.255,NaN,NaN,NaN,NaN,NaN
83,NaN,NaN,NaN,4.0,4000,0.294892,0.9350,0.911373,3.5155,568.901,35.556,NaN,NaN,NaN,NaN,NaN
104,NaN,NaN,NaN,5.0,5000,0.383528,0.9310,0.902016,3.5718,559.937,34.996,NaN,NaN,NaN,NaN,NaN
125,NaN,NaN,NaN,6.0,6000,0.325411,0.9370,0.911572,3.4505,579.623,36.226,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,7.0,7000,0.351091,0.9405,0.918494,3.4743,575.660,35.979,NaN,NaN,NaN,NaN,NaN
167,NaN,NaN,NaN,8.0,8000,0.406794,0.9420,0.921095,3.4889,573.245,35.828,NaN,NaN,NaN,NaN,NaN


In [26]:
from sklearn.metrics import classification_report

output = trainer.predict(tokenized_dataset["validation"])
predictions = np.argmax(output.predictions, axis=-1)

label_names = dataset["train"].features["label"].names

print(classification_report(
    output.label_ids,
    predictions,
    labels=list(range(len(label_names))),
    target_names=label_names,
    digits=3,
    zero_division=0,
))

              precision    recall  f1-score   support

     sadness      0.965     0.965     0.965       550
         joy      0.960     0.959     0.959       704
        love      0.878     0.888     0.883       178
       anger      0.944     0.927     0.936       275
        fear      0.883     0.929     0.906       212
    surprise      0.919     0.840     0.877        81

    accuracy                          0.942      2000
   macro avg      0.925     0.918     0.921      2000
weighted avg      0.942     0.942     0.942      2000



In [27]:
results = dataset["validation"].to_pandas()

results["true_emotion"] = [
    label_names[i] for i in output.label_ids
]
results["predicted_emotion"] = [
    label_names[i] for i in predictions
]

errors = results[
    results["true_emotion"] != results["predicted_emotion"]
]

print("Mistakes:", len(errors))

errors[["text", "true_emotion", "predicted_emotion"]].sample(
    n=min(10, len(errors)),
    random_state=42,
)

Mistakes: 116


,text,true_emotion,predicted_emotion
1579,ive seen a lot of seizures but never this many...,fear,sadness
83,i feel stressed or my family is being negative...,sadness,anger
684,i feel strongly that by supporting because i a...,joy,love
671,i feel so blessed to have known both,joy,love
200,i really have much of a clue how my ex actuall...,anger,sadness
792,i know she feels helpless but that kiss that c...,sadness,fear
1939,i feel accepted and loved and forgiven the gra...,joy,love
603,i have had several new members tell me how com...,joy,love
1251,i feel my sweet boy traveling this difficult r...,love,joy
218,i feel overwhelmed by my circumstance in all o...,surprise,fear


In [28]:
print(trainer.args.learning_rate)

5e-05


In [29]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

# Preserve the previous run's result
baseline_best_f1 = trainer.state.best_metric
baseline_checkpoint = trainer.state.best_model_checkpoint

set_seed(42)

label_names = dataset["train"].features["label"].names
id2label = dict(enumerate(label_names))
label2id = {name: i for i, name in id2label.items()}

# Start from pretrained BERT, not the previously fine-tuned model
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

train_args_v2 = TrainingArguments(
    output_dir="./results-lr-2e-5",
    learning_rate=2e-5,
    weight_decay=0.01,

    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,

    logging_steps=50,
    report_to="none",
    seed=42,
)

trainer_v2 = Trainer(
    model=model_v2,
    args=train_args_v2,
    data_collator=collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_v2.train()

print(f"Baseline best validation macro-F1: {baseline_best_f1:.4f}")
print(f"New run best validation macro-F1:  {trainer_v2.state.best_metric:.4f}")
print("New best checkpoint:", trainer_v2.state.best_model_checkpoint)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 638.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.231650,0.194167,0.925500,0.902594
2,0.114056,0.165272,0.937000,0.909786
3,0.090383,0.156282,0.940500,0.916223
4,0.095260,0.196924,0.939500,0.917190
5,0.060958,0.249978,0.940000,0.917596
6,0.041281,0.298842,0.933000,0.905990
7,0.022277,0.338695,0.939500,0.916237
8,0.026343,0.345796,0.937000,0.912681


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

Baseline best validation macro-F1: 0.9211
New run best validation macro-F1:  0.9176
New best checkpoint: ./results-lr-2e-5\checkpoint-5000


In [30]:
from transformers import set_seed

set_seed(42)

model_v3 = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(label_names),
    id2label=dict(enumerate(label_names)),
    label2id={name: i for i, name in enumerate(label_names)},
    classifier_dropout=0.2,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 565.20it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [31]:
from dataclasses import replace

train_args_v3 = replace(
    train_args_v2,
    output_dir="./results-dropout-02",
)

trainer_v3 = Trainer(
    model=model_v3,
    args=train_args_v3,
    data_collator=collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_v3.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.240930,0.191743,0.930000,0.904348
2,0.156836,0.159997,0.938500,0.913954
3,0.085019,0.164666,0.941000,0.919148
4,0.118574,0.194338,0.937500,0.915658
5,0.055482,0.260112,0.937500,0.915406
6,0.042075,0.301913,0.937500,0.912718


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=6000, training_loss=0.1561299977997939, metrics={'train_runtime': 1514.9406, 'train_samples_per_second': 105.615, 'train_steps_per_second': 6.601, 'total_flos': 2318165650211904.0, 'train_loss': 0.1561299977997939, 'epoch': 6.0})

In [32]:
from dataclasses import replace
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

set_seed(42)

# Fresh pretrained weights, same configuration as the dropout run
model_v4 = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(label_names),
    id2label=dict(enumerate(label_names)),
    label2id={name: i for i, name in enumerate(label_names)},
    classifier_dropout=0.2,
)

# Freeze embeddings
for parameter in model_v4.bert.embeddings.parameters():
    parameter.requires_grad = False

# Freeze encoder blocks 0–5; blocks 6–11 remain trainable
for layer in model_v4.bert.encoder.layer[:6]:
    for parameter in layer.parameters():
        parameter.requires_grad = False

total = sum(p.numel() for p in model_v4.parameters())
trainable = sum(
    p.numel() for p in model_v4.parameters() if p.requires_grad
)
print(f"Trainable parameters: {trainable:,} / {total:,}")

# Copy settings; only freezing differs from the previous experiment
train_args_v4 = replace(
    train_args_v3,
    output_dir="./results-frozen-6",
)

trainer_v4 = Trainer(
    model=model_v4,
    args=train_args_v4,
    data_collator=collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_v4.train()

print(f"Previous best macro-F1: {trainer_v3.state.best_metric:.4f}")
print(f"Frozen best macro-F1:   {trainer_v4.state.best_metric:.4f}")

# Metrics for the restored best checkpoint
print(trainer_v4.evaluate())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 648.40it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Trainable parameters: 43,122,438 / 109,486,854


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.353089,0.313738,0.893000,0.861490


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.66it/s]


KeyboardInterrupt: 

In [34]:
print(trainer_v4.args.device)
print(next(trainer_v4.model.parameters()).device)

cuda:0
cuda:0
